# SpaNCy-Shift CFM Explorer

OT-Conditional Flow Matching as Stage 2 batch correction for CyCIF multiplexed imaging.

**Stage 1 (analytic):** Per-marker shift toward KL-medoid reference → `normalized_base`. kBET ≈ 0.631.

**Stage 2 (OT-CFM):** `FlowMLP` conditioned on source batch learns a velocity field transporting
cells toward the reference-batch distribution. Training: mini-batch OT coupling (Hungarian algorithm
on 256×256 L2 cost matrix) + MSE velocity loss. Inference: Euler ODE integration t=0 → t=1.

No torch-geometric. No adversarial training. No bimodal masking needed — velocity field learns
the full joint distribution including bimodal structure naturally.

**Sections:**
0. Colab Setup
1. Load & Inspect Data
2. Bimodal Marker Detection
2b. Reference Sample Selection
3. Training (Stage 2 OT-CFM)
3b. n_steps Sweep (correction strength calibration)
4. Inference & Histogram Inspection
5. Batch adj-R² Diagnostics
6. Positive Population Preservation Check
7. Histogram Comparison PDF
8. kBET Evaluation

## 0. Colab Setup

Run first on Google Colab. Installs dependencies and uploads both `.py` files.

**Note:** No torch-geometric needed for CFM — pure torch + scipy.

In [ ]:
!pip install -q anndata scanpy pegasuspy pegasusio
print('Done.')

In [ ]:
# Upload BOTH spancy_shift.py AND spancy_shift_cfm.py
from google.colab import files
uploaded = files.upload()

import os
assert os.path.exists('spancy_shift.py'), 'spancy_shift.py not found'
assert os.path.exists('spancy_shift_cfm.py'), 'spancy_shift_cfm.py not found'
print(f"spancy_shift.py    ({os.path.getsize('spancy_shift.py'):,} bytes)")
print(f"spancy_shift_cfm.py ({os.path.getsize('spancy_shift_cfm.py'):,} bytes)")

In [ ]:
import importlib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import matplotlib.pyplot as plt

import spancy_shift
import spancy_shift_cfm
from spancy_shift import (
    load_adata, log1p_scale,
    detect_bimodal_markers, find_best_sample_per_marker,
    shift_normalize_per_marker,
    positive_population_table, per_marker_batch_r2,
)
from spancy_shift_cfm import (
    FlowMLP, BatchBalancedSampler, ot_couple,
    train_cfm, normalize_adata_cfm,
)

# After re-uploading files:
# importlib.reload(spancy_shift); importlib.reload(spancy_shift_cfm)
# from spancy_shift import *; from spancy_shift_cfm import *

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# Download PRAD dataset
!wget -q https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad
print('Download complete.')

## 1. Load & Inspect Data

In [ ]:
DATA_PATH = 'PRAD_anndata.h5ad'

adata = load_adata(DATA_PATH)
print(adata)
print(f"\nMarkers ({adata.n_vars}): {list(adata.var_names)}")
print(f"Batches:  {sorted(adata.obs['batch_id'].unique().tolist())}")
print(f"Samples:  {sorted(adata.obs['sample_id'].unique().tolist())}")
print(f"\nobs columns: {list(adata.obs.columns)}")

marker_names = list(adata.var_names)
X_raw = np.asarray(adata.X.toarray() if sp.issparse(adata.X) else adata.X)
batch_vals = adata.obs['batch_id'].values
unique_batches = sorted(adata.obs['batch_id'].unique())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
axes[0].boxplot([X_raw[:, i] for i in range(X_raw.shape[1])], labels=marker_names, vert=True)
axes[0].set_xticklabels(marker_names, rotation=90, fontsize=8)
axes[0].set_title('Raw marker distributions (all cells)')
axes[0].set_ylabel('Intensity')
for b in unique_batches:
    mask = batch_vals == b
    axes[1].plot(X_raw[mask].mean(axis=0), marker='o', markersize=3, label=str(b), alpha=0.8)
axes[1].set_xticks(range(len(marker_names)))
axes[1].set_xticklabels(marker_names, rotation=90, fontsize=8)
axes[1].set_title('Per-batch mean intensity')
axes[1].legend(fontsize=6, ncol=2)
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Bimodal Marker Detection (preview on raw data)

Same as spancy_shift — CFM does NOT need explicit bimodal masking.
The velocity field learns the full joint distribution including bimodal structure.
Shown here for reference only.

In [ ]:
BIMODAL_MIN_BATCH_FRAC = 0.5

X_log_preview = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)
batch_codes_preview = adata.obs['batch_id'].astype('category').cat.codes.values.astype('int64')

marker_is_bimodal, thresholds = detect_bimodal_markers(
    X_log_preview, marker_names,
    batch_codes=batch_codes_preview,
    min_prominence_frac=0.05,
    bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
)

print(f"Bimodal markers ({marker_is_bimodal.sum()}):")
for i, name in enumerate(marker_names):
    if marker_is_bimodal[i]:
        print(f"  {name:>12s}  threshold={thresholds[i]:.3f}")
print(f"\nUnimodal markers ({(~marker_is_bimodal).sum()}):")
for i, name in enumerate(marker_names):
    if not marker_is_bimodal[i]:
        print(f"  {name}")
print("\nNote: CFM learns the full distribution — no special handling for bimodal markers.")

## 2b. Reference Sample Selection

For each marker, find the medoid sample (lowest mean pairwise KL divergence).
The OT coupling targets cells from whichever batch contains the most reference samples.

In [ ]:
ref_sample_per_marker = find_best_sample_per_marker(adata)

ref_df = pd.DataFrame(
    [(m, s) for m, s in ref_sample_per_marker.items()],
    columns=['marker', 'reference_sample'],
)
print('Per-marker reference sample:\n')
print(ref_df.to_string(index=False))

ref_counts = ref_df['reference_sample'].value_counts()
print(f'\nDistinct reference samples: {len(ref_counts)}')
print(ref_counts.to_string())

fig, ax = plt.subplots(figsize=(14, 3))
sample_order = sorted(adata.obs['sample_id'].unique())
ref_matrix = [sum(v == s for v in ref_sample_per_marker.values()) for s in sample_order]
ax.bar(range(len(sample_order)), ref_matrix, color='steelblue', alpha=0.85)
ax.set_xticks(range(len(sample_order)))
ax.set_xticklabels(sample_order, rotation=45, ha='right')
ax.set_ylabel('# markers using as reference')
ax.set_title('Reference sample selection (medoid)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## 3. Stage 2 Training (OT-CFM)

Trains `FlowMLP` on Stage 1 output (`normalized_base`).

**Inside `train_cfm()`:**
1. Runs `shift_normalize_per_marker()` → `X_base` (Stage 1)
2. Fits `RobustScaler` on `log1p(X_base)` → `X_scaled`
3. Identifies reference batch (majority vote over per-marker reference samples)
4. For each training step:
   - Sample source cells from all batches (batch-balanced)
   - OT coupling: Hungarian algorithm on 256×256 L2 cost matrix → phenotype-matched pairs
   - Interpolate at random t ∈ [0,1]: `x_t = (1-t)*x_0 + t*x_1 + σ*noise`
   - Predict velocity toward x_1; target = `x_1 - x_0`; loss = MSE

**Inference:** Euler ODE from t=0 to t=1 (n_steps Euler steps).
`n_steps` controls correction strength: 5 = very conservative, 50 = full transport.

In [ ]:
N_STEPS = 20    # Euler ODE steps at inference — tune in sec3b
N_EPOCHS = 10   # quick test; use 50-100 for production

model, trained_scaler, trained_ref, history = train_cfm(
    adata,
    n_epochs=N_EPOCHS,
    lr=1e-3,
    device_str=DEVICE,
    warmup_epochs=2,
    n_per_batch=256,
    ot_samples=256,
    sigma_min=0.01,
    hidden=512,
    n_layers=6,
    bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
    ref_sample_per_marker=ref_sample_per_marker,
)

print(f'\nTraining complete.')
print(f'  Reference batch: {model._ref_batch_name} (code {model._ref_batch_code})')
print(f'  Final loss={history["loss"][-1]:.6f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history['loss'], color='k', linewidth=1.5)
axes[0].set_title('OT-CFM Velocity Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['lr'], color='purple', linewidth=1.5)
axes[1].set_title('Learning Rate')
axes[1].set_xlabel('Epoch')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Stage 2 Training — FlowMLP (OT-CFM)', fontsize=13)
plt.tight_layout()
plt.show()

## 3b. n_steps Sweep — Correction Strength Calibration

OT-CFM's correction strength is controlled by `n_steps` (number of Euler integration steps).
Fewer steps = cells moved less = more conservative = better biology preservation.
More steps = cells moved more = stronger alignment = better kBET.

This sweep helps find the right balance before running full kBET evaluation.
Check ECAD histogram — it must remain bimodal at the chosen n_steps.

In [ ]:
N_STEPS_SWEEP = [5, 20, 50]   # conservative → aggressive
BIOLOGY_MARKER = 'ECAD'       # bimodal marker — must stay bimodal
CHECK_MARKERS = ['ECAD', 'CD45', 'ChromA']  # markers to inspect

results_sweep = {}
for n_steps_val in N_STEPS_SWEEP:
    print(f'\n--- n_steps={n_steps_val} ---')
    adata_sw = normalize_adata_cfm(
        adata, model, trained_scaler, trained_ref,
        n_steps=n_steps_val,
        inference_batch_size=4096,
        device_str=DEVICE,
        layer_name=f'norm_nsteps{n_steps_val}',
        keep_base_layer=True,
    )
    results_sweep[n_steps_val] = np.asarray(adata_sw.layers[f'norm_nsteps{n_steps_val}'])
print('\nSweep done.')

In [ ]:
# Histogram comparison across n_steps values
n_rows = len(CHECK_MARKERS)
n_cols = len(N_STEPS_SWEEP) + 1  # +1 for Stage 1 baseline
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))

for row, mname in enumerate(CHECK_MARKERS):
    if mname not in marker_names:
        continue
    m_idx = marker_names.index(mname)

    for col, (label, data) in enumerate(
        [('Stage 1 (base)', np.asarray(adata_sw.layers['normalized_base']))]
        + [(f'CFM n_steps={ns}', results_sweep[ns]) for ns in N_STEPS_SWEEP]
    ):
        ax = axes[row, col]
        for b in unique_batches:
            mask = batch_vals == b
            vals = np.log1p(np.clip(data[mask, m_idx], 0, None))
            ax.hist(vals, bins=80, alpha=0.4, density=True, label=str(b))
        ax.set_title(f'{mname} — {label}', fontsize=9)
        ax.set_xlabel('log1p')
        if col == 0:
            ax.set_ylabel('Density')
        if row == 0 and col == 0:
            ax.legend(fontsize=6, ncol=2, title='Batch')

plt.suptitle(f'n_steps sweep — key markers', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nChoose n_steps where ECAD remains bimodal and batches overlap visually.')
print(f'Recommended starting point: N_STEPS = 20')

## 4. Inference & Output Inspection

Run full inference with chosen `N_STEPS`. Produces both Stage 1 and Stage 2 layers.

In [ ]:
# Run inference with the chosen N_STEPS
adata_norm = normalize_adata_cfm(
    adata, model, trained_scaler, trained_ref,
    n_steps=N_STEPS,
    inference_batch_size=4096,
    device_str=DEVICE,
    layer_name='normalized',
    keep_base_layer=True,
)

X_base = np.asarray(adata_norm.layers['normalized_base'])
X_norm = np.asarray(adata_norm.layers['normalized'])

print(f'Stage 1 (normalized_base): min={X_base.min():.4f}  max={X_base.max():.4f}  mean={X_base.mean():.4f}')
print(f'Stage 2 (normalized):      min={X_norm.min():.4f}  max={X_norm.max():.4f}  mean={X_norm.mean():.4f}')
print(f'\nDelta stats (Stage 2 - Stage 1):')
delta_arr = X_norm - X_base
for i, mname in enumerate(marker_names):
    print(f'  {mname:>12s}  mean_delta={delta_arr[:, i].mean():.4f}  std_delta={delta_arr[:, i].std():.4f}')

In [ ]:
# Compare raw vs Stage 1 vs Stage 2 for bimodal markers + a few unimodal
bimodal_names = [marker_names[i] for i in range(len(marker_names)) if marker_is_bimodal[i]]
plot_markers = (bimodal_names[:3] + [m for m in marker_names if m not in bimodal_names][:2])
if not plot_markers:
    plot_markers = marker_names[:5]

layers_to_show = [
    (None,             'Raw'),
    ('normalized_base', 'Stage 1 (analytic)'),
    ('normalized',      'Stage 2 (OT-CFM)'),
]

fig, axes = plt.subplots(len(plot_markers), len(layers_to_show),
                          figsize=(6 * len(layers_to_show), 4 * len(plot_markers)))
if len(plot_markers) == 1:
    axes = axes.reshape(1, -1)

for row, mname in enumerate(plot_markers):
    m_idx = marker_names.index(mname)
    for col, (layer_key, title) in enumerate(layers_to_show):
        ax = axes[row, col]
        data = X_raw if layer_key is None else np.asarray(adata_norm.layers[layer_key])
        for b in unique_batches:
            mask = batch_vals == b
            vals = np.log1p(np.clip(data[mask, m_idx], 0, None))
            ax.hist(vals, bins=80, alpha=0.4, density=True, label=str(b))
        ax.set_title(f'{mname} — {title}')
        ax.set_xlabel('log1p intensity')
        if col == 0:
            ax.set_ylabel('Density')
        if row == 0 and col == 0:
            ax.legend(fontsize=6, title='Batch', ncol=2)

plt.suptitle(f'Marker Distributions: Raw → Stage 1 → Stage 2 (CFM, n_steps={N_STEPS})', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Batch adj-R² Diagnostics

Adjusted R² for each marker regressed on batch labels. Lower = batch effect better removed.

In [ ]:
batch_arr = adata_norm.obs['batch_id'].values
X_base_arr = np.asarray(adata_norm.layers['normalized_base'])
X_norm_arr = np.asarray(adata_norm.layers['normalized'])

r2_raw   = per_marker_batch_r2(np.log1p(np.clip(X_raw, 0, None)),       batch_arr, marker_names)
r2_base  = per_marker_batch_r2(np.log1p(np.clip(X_base_arr, 0, None)),  batch_arr, marker_names)
r2_cfm   = per_marker_batch_r2(np.log1p(np.clip(X_norm_arr, 0, None)),  batch_arr, marker_names)

r2_compare = (
    r2_raw .rename(columns={'adj_r2': 'raw'})
    .merge(r2_base.rename(columns={'adj_r2': 'stage1_base'}), on='marker')
    .merge(r2_cfm .rename(columns={'adj_r2': 'stage2_cfm'}),  on='marker')
)

print('Per-Marker Batch adj-R² (lower = better):\n')
print(r2_compare.to_string(index=False, float_format='%.4f'))
print(f'\nMean adj-R²:')
for col in ['raw', 'stage1_base', 'stage2_cfm']:
    print(f'  {col:>14s}: {r2_compare[col].mean():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
x = np.arange(len(r2_compare))
w = 0.25
ax.bar(x - w, r2_compare['raw'],          w, label='Raw',            color='salmon',      alpha=0.85)
ax.bar(x,     r2_compare['stage1_base'],  w, label='Stage 1 (base)', color='steelblue',   alpha=0.85)
ax.bar(x + w, r2_compare['stage2_cfm'],   w, label='Stage 2 (CFM)',  color='forestgreen', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(r2_compare['marker'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Adjusted R²')
ax.set_title('Per-Marker Batch adj-R² — lower is better')
ax.axhline(0.05, color='red', linestyle='--', alpha=0.5, label='Target (0.05)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Positive Population Preservation Check

% positive cells per marker per sample using Otsu's threshold. Target: |Δ| < 5%.

In [ ]:
pop_table_base = positive_population_table(
    adata_norm, norm_layer='normalized_base', sample_col='sample_id'
)
pop_table = positive_population_table(
    adata_norm, norm_layer='normalized', sample_col='sample_id'
)

for label, pt in [('Stage 1 (base)', pop_table_base), ('Stage 2 CFM (norm)', pop_table)]:
    summary = pt.groupby('marker').agg(
        mean_pct_raw=('pct_pos_raw', 'mean'),
        mean_pct_norm=('pct_pos_norm', 'mean'),
        mean_abs_delta=('delta', lambda x: x.abs().mean()),
        max_abs_delta=('delta', lambda x: x.abs().max()),
    ).round(2)
    print(f'Positive Population Preservation ({label} vs raw):')
    print('=' * 65)
    print(summary.to_string())
    print(f'\nOverall mean |delta|: {pt["delta"].abs().mean():.2f}%')
    print(f'Overall max  |delta|: {pt["delta"].abs().max():.2f}%')
    print()

print('Target: |delta| < 5% per marker  (UniFORM: -3.4% mean but large per-marker distortions)')

# ── Incremental delta: how much CFM adds on top of Stage 1 ──────────────────
s1  = pop_table_base.groupby('marker')['pct_pos_norm'].mean().rename('stage1')
cfm = pop_table     .groupby('marker')['pct_pos_norm'].mean().rename('cfm')
raw = pop_table     .groupby('marker')['pct_pos_raw' ].mean().rename('raw')
incr = pd.concat([raw, s1, cfm], axis=1)
incr['s1_vs_raw']  = (incr['stage1'] - incr['raw']).round(1)
incr['cfm_vs_s1']  = (incr['cfm']    - incr['stage1']).round(1)
incr['cfm_vs_raw'] = (incr['cfm']    - incr['raw']).round(1)

print('\n── CFM Incremental Delta (Stage 2 vs Stage 1) ──────────────────────────')
print('   Large |delta vs raw| is Stage 1 correction. CFM adds the incremental column.')
print(f'   Mean |CFM incremental|: {incr["cfm_vs_s1"].abs().mean():.2f}%')
print()
print(incr[['raw','stage1','cfm','s1_vs_raw','cfm_vs_s1']].to_string(float_format='%.1f'))

In [ ]:
pivot = pop_table.pivot(index='marker', columns='sample', values='delta')
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(pivot.values, cmap='RdBu_r', vmin=-10, vmax=10, aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)
plt.colorbar(im, label='Δ % positive (norm − raw)')
ax.set_title('Positive Population Change: CFM Stage 2 vs Raw')
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.patches import Patch

METHOD_ORDER  = ['Raw', 'Stage 1', 'Stage 2 CFM']
METHOD_COLORS = {
    'Raw':         '#888888',
    'Stage 1':     '#4878d0',
    'Stage 2 CFM': '#ff7f0e',
}

# Build unified long-format table
frames = []
for method_name, pt in [('Stage 1', pop_table_base), ('Stage 2 CFM', pop_table)]:
    tmp = pt[['marker', 'sample', 'pct_pos_norm']].rename(columns={'pct_pos_norm': 'pct_pos'}).copy()
    tmp['method'] = method_name
    frames.append(tmp)
raw_tmp = pop_table[['marker', 'sample', 'pct_pos_raw']].rename(columns={'pct_pos_raw': 'pct_pos'}).copy()
raw_tmp['method'] = 'Raw'
frames.append(raw_tmp)
all_pop = pd.concat(frames, ignore_index=True)

# Violin grid: one subplot per marker
all_markers_sorted = sorted(all_pop['marker'].unique().tolist())
n_markers = len(all_markers_sorted)
ncols = 4
nrows = (n_markers + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 4), sharey=False)
axes = axes.ravel()

for i, marker in enumerate(all_markers_sorted):
    ax = axes[i]
    sub = all_pop[all_pop['marker'] == marker]
    data_by_method = [sub[sub['method'] == m]['pct_pos'].dropna().values for m in METHOD_ORDER]

    parts = ax.violinplot(
        [d if len(d) > 0 else np.array([0.0]) for d in data_by_method],
        positions=np.arange(len(METHOD_ORDER)),
        showmedians=True, showextrema=True, widths=0.65,
    )
    for body, m in zip(parts['bodies'], METHOD_ORDER):
        body.set_facecolor(METHOD_COLORS[m])
        body.set_alpha(0.75)
        body.set_edgecolor('none')
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(2)
    for key in ('cmaxes', 'cmins'):
        parts[key].set_visible(False)
    parts['cbars'].set_linewidth(0.8)
    parts['cbars'].set_color('gray')

    for j, d in enumerate(data_by_method):
        if len(d) > 0:
            ax.scatter(j, np.nanmean(d), s=35, color='white',
                       edgecolor='black', linewidth=1.0, zorder=5)

    is_bim = marker in bimodal_names
    ax.set_title(marker, fontsize=9, fontweight='bold',
                 color='navy' if is_bim else 'black')
    ax.set_xticks(np.arange(len(METHOD_ORDER)))
    ax.set_xticklabels(METHOD_ORDER, rotation=45, ha='right', fontsize=6.5)
    ax.set_ylabel('% Positive', fontsize=7)
    ax.grid(True, alpha=0.3, axis='y')
    y_max = float(sub['pct_pos'].max()) if len(sub) > 0 else 5.0
    ax.set_ylim(-2, max(y_max, 5.0) + 5)

for i in range(n_markers, len(axes)):
    axes[i].set_visible(False)

legend_patches = [Patch(facecolor=METHOD_COLORS[m], alpha=0.8, label=m) for m in METHOD_ORDER]
fig.legend(handles=legend_patches, loc='lower right', ncol=len(METHOD_ORDER),
           fontsize=9, bbox_to_anchor=(0.99, 0.01), frameon=True)
plt.suptitle(
    'Positive Population (% positive cells) per Marker — Raw vs Stage 1 vs Stage 2 CFM\n'
    'Violin = sample distribution  ·  White dot = mean  ·  Black line = median\n'
    'Navy titles = bimodal markers',
    fontsize=11,
)
plt.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.show()

# Mean % positive summary table
pivot_mean = (
    all_pop.groupby(['marker', 'method'])['pct_pos']
    .mean().unstack('method')[METHOD_ORDER].round(1)
)
print('\nMean % Positive Cells per Marker (across samples):')
print('=' * 65)
print(pivot_mean.to_string())

raw_ref = all_pop[all_pop['method'] == 'Raw']['pct_pos'].mean()
print('\nOverall mean % positive (all markers × samples):')
for m in METHOD_ORDER:
    overall = all_pop[all_pop['method'] == m]['pct_pos'].mean()
    delta   = overall - raw_ref
    sign    = '+' if delta >= 0 else ''
    print(f'  {m:>14s}: {overall:.2f}%   (Δ from Raw: {sign}{delta:.2f}%)')

## 7. Histogram Comparison PDF

Per-sample histogram PDFs for all markers. Saved to `histograms_cfm/`.

In [ ]:
import os
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import colormaps

save_dir = 'histograms_cfm'
os.makedirs(save_dir, exist_ok=True)

layers_to_plot = [
    ('normalized_base', 'Stage 1 (analytic)'),
    ('normalized',      f'Stage 2 (OT-CFM, n_steps={N_STEPS})'),
]

sample_col = 'sample_id' if 'sample_id' in adata_norm.obs.columns else 'batch_id'
sample_ids = adata_norm.obs[sample_col].astype(str).values
unique_samples = sorted(np.unique(sample_ids).tolist())
cmap = colormaps.get_cmap('tab20')
colors = {s: cmap(i % 20) for i, s in enumerate(unique_samples)}

pdf_path = os.path.join(save_dir, 'cfm_histograms.pdf')
rows_per_page = 4

with PdfPages(pdf_path) as pdf:
    page_markers = []
    for i, mname in enumerate(marker_names):
        page_markers.append(mname)
        if len(page_markers) == rows_per_page or i == len(marker_names) - 1:
            fig, axes = plt.subplots(len(page_markers), len(layers_to_plot),
                                     figsize=(7 * len(layers_to_plot), 4 * len(page_markers)))
            if len(page_markers) == 1:
                axes = axes.reshape(1, -1)
            for row, mname in enumerate(page_markers):
                m_idx = marker_names.index(mname)
                for col, (layer_key, label) in enumerate(layers_to_plot):
                    ax = axes[row, col]
                    X_col = np.asarray(adata_norm.layers[layer_key][:, m_idx]).ravel()
                    X_log = np.log1p(np.clip(X_col, 0, None))
                    for s in unique_samples:
                        mask = sample_ids == s
                        c, e = np.histogram(X_log[mask], bins=80)
                        ax.plot(e[:-1], c, color=colors[s], linewidth=1.5, alpha=0.7, label=s)
                    bim_tag = ' [BIMODAL]' if marker_is_bimodal[m_idx] else ''
                    ax.set_title(f'{mname}{bim_tag} — {label}', fontsize=9)
                    ax.set_xlabel('log1p intensity', fontsize=8)
                    if col == 0:
                        ax.set_ylabel('Count', fontsize=8)
                    if row == 0 and col == 0:
                        ax.legend(fontsize=5, ncol=3, frameon=False)
                    ax.spines[['top', 'right']].set_visible(False)
            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)
            page_markers = []

print(f'PDF saved to: {pdf_path}')

## 8. kBET Evaluation

kBET acceptance rate for 5 clinical groups pairing samples from different batches.

- **Higher kBET = better** (batches well-mixed in local neighborhoods)
- Stage 1 baseline: ~0.631
- UniFORM baseline: 0.631
- Target: Stage 2 CFM > 0.631 with positive population |Δ| < 5%

In [ ]:
import pegasus as pg
import pegasusio as pgio

def subset_ab(adata, sample, batch):
    mask = (adata.obs['sample_id'] == sample) & (adata.obs['batch_id'] == batch)
    return adata[mask, :].copy()

GROUP_DEFS = {
    'g1': [('PRAD-02', 'batch1'), ('PRAD-14', 'batch7')],
    'g2': [('PRAD-01', 'batch4'), ('PRAD-02', 'batch1')],
    'g3': [('PRAD-05', 'batch1'), ('PRAD-12', 'batch2')],
    'g4': [('PRAD-07', 'batch2'), ('PRAD-19', 'batch6')],
    'g5': [('PRAD-02', 'batch1'), ('PRAD-07', 'batch2')],
}

EVAL_LAYERS = ['normalized_base', 'normalized']
print(f'Evaluating: {EVAL_LAYERS}')
print()
print('Baselines:')
print('  Stage 1 (analytic):              ~0.631')
print('  UniFORM:                          0.631')
print('  SpaNCy-GNN ensemble hybrid:       0.574')
print(f'  Stage 2 CFM (n_steps={N_STEPS}):   ???  (target > 0.631)')

In [ ]:
all_kbet = {}

for layer_name in EVAL_LAYERS:
    print(f'\n{"-"*55}\n  {layer_name}\n{"-"*55}')
    adata_kbet = adata_norm.copy()
    adata_kbet.X = adata_norm.layers[layer_name].copy()

    layer_res = {}
    for gname, pairs in GROUP_DEFS.items():
        try:
            parts = [subset_ab(adata_kbet, s, b) for s, b in pairs]
            adata_g = parts[0].concatenate(parts[1], batch_key=None)
            mmdata = pgio.MultimodalData(adata_g)
            pg.qc_metrics(mmdata)
            pg.filter_data(mmdata)
            pg.identify_robust_genes(mmdata)
            pg.pca(mmdata, features=None, n_components=20)
            pg.neighbors(mmdata)
            pg.umap(mmdata)
            stat, pval, score = pg.calc_kBET(mmdata, attr='batch_id', rep='umap')
            layer_res[gname] = {'kBET': score, 'chi2': stat, 'p': pval}
            print(f'  {gname}: kBET={score:.4f}  chi2={stat:.4f}  p={pval:.4f}')
        except Exception as e:
            print(f'  {gname}: FAILED — {e}')
            layer_res[gname] = {'kBET': float('nan'), 'chi2': float('nan'), 'p': float('nan')}

    all_kbet[layer_name] = layer_res

print('\nDone.')

In [ ]:
print('=' * 70)
print('kBET Summary (mean across groups — higher = better)')
print('=' * 70)

summary_rows = []
for layer_name, res in all_kbet.items():
    kbets = [v['kBET'] for v in res.values() if not np.isnan(v['kBET'])]
    chi2s = [v['chi2'] for v in res.values() if not np.isnan(v['chi2'])]
    if kbets:
        row = {
            'layer': layer_name,
            'mean_kBET': float(np.mean(kbets)),
            'mean_chi2': float(np.mean(chi2s)),
            'n_groups': f'{len(kbets)}/{len(res)}',
        }
        summary_rows.append(row)
        print(f"  {layer_name:>20s}  kBET={row['mean_kBET']:.4f}  "
              f"chi2={row['mean_chi2']:.4f}  ({row['n_groups']} groups)")

print()
for gname, res in all_kbet.get('normalized', {}).items():
    if not np.isnan(res['kBET']):
        print(f"{gname}: kBET={res['kBET']:.4f}  chi2={res['chi2']:.4f}  p={res['p']:.4f}")

print()
print('Baselines:')
print('  Stage 1 expected:            kBET~0.631')
print('  UniFORM:                     kBET=0.631')
print(f'  Stage 2 CFM (n_steps={N_STEPS}):  kBET=???  (target > 0.631)')

df_kbet = pd.DataFrame(summary_rows)

if len(df_kbet) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    colors = ['steelblue', 'forestgreen'][:len(df_kbet)]
    axes[0].bar(df_kbet['layer'], df_kbet['mean_kBET'], color=colors, alpha=0.85)
    axes[0].axhline(0.631, color='red', linestyle='--', linewidth=1.5, label='UniFORM 0.631')
    axes[0].set_ylabel('Mean kBET acceptance rate')
    axes[0].set_title('kBET (higher = better)')
    axes[0].set_xticklabels(df_kbet['layer'], rotation=20, ha='right', fontsize=9)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')
    axes[1].bar(df_kbet['layer'], df_kbet['mean_chi2'], color=colors, alpha=0.85)
    axes[1].set_ylabel('Mean chi-square')
    axes[1].set_title('Chi-square (lower = better)')
    axes[1].set_xticklabels(df_kbet['layer'], rotation=20, ha='right', fontsize=9)
    axes[1].grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()